In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpalzfu59u".


In [2]:
%%cuda
#include <iostream>

// THE KERNEL
__global__ void scaleArray(float* data, float factor, int n) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;

    if (i < n) {
        // Simple element-wise operation
        data[i] = data[i] * factor;
    }
}

int main() {
    int n = 50000;
    float scaleFactor = 5.0f;
    size_t size = n * sizeof(float);

    float *h_data = (float*)malloc(size);
    for(int i=0; i<n; i++) h_data[i] = 2.0f; // Initialize with 2.0

    float *d_data;
    cudaMalloc(&d_data, size);
    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);

    // Launch: 256 threads per block is the "Sweet Spot" for most GPUs
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    scaleArray<<<blocks, threads>>>(d_data, scaleFactor, n);

    cudaMemcpy(h_data, d_data, size, cudaMemcpyDeviceToHost);

    // Expected: 2.0 * 5.0 = 10.0
    std::cout << "Data[0]: " << h_data[0] << " (Expected 10.0)" << std::endl;

    cudaFree(d_data);
    free(h_data);
    return 0;
}

Data[0]: 10 (Expected 10.0)

